# v5: Hindi-only control to 90 phases (GPU 1)
Hindi in every phase. Phases 19-90 continue the v4b control.
* **Same recipe as v4b**: same model (QK-norm), same per-phase schedule (fresh AdamW, warmup, stable until plateau, cooldown), same divergence guard (val > 1.15x best twice → retry at half the learning rate, max 2), z-loss off, script-restricted loss and decoding.
* **Resumes** from the last saved phase (phase 18) and continues to **phase 90 = 30 cycles**. Nothing restarts from scratch.
* **Every phase** saves a model-only checkpoint `runs/<run>/ckpt/pXXX_<lang>.pt` (~270 MB) and appends to `phases.jsonl` / `log.jsonl`.
* **Best models**: `runs/<run>/best/best_<lang>.pt` = the checkpoint with the highest chrF on that language (`best.json` says which phase).
* **HF backup**: each new checkpoint, the logs and any new best model are uploaded to your HF repo in the background, so a dead instance loses at most one phase.
* Value vectors are hookable (`probe_points(model)` in common_v5.py): used later by the analysis, not during training.
* Rerunning the notebook at any time resumes where it stopped.

In [1]:
import os, sys
os.environ["CUDA_VISIBLE_DEVICES"] = "1"      # run this cell FIRST in a fresh kernel
REPO = "/workspace/shreshth"
os.chdir(REPO); sys.path.insert(0, REPO)
from common_v5 import *
print("device:", DEVICE, "|", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "CPU")

device: cuda | NVIDIA GeForce RTX 4090


In [2]:
# Get everything this run needs onto this machine (skips what is already here)
from huggingface_hub import HfApi, snapshot_download
RUN = "v4b_only_hi_s0"
HF_REPO = HfApi().whoami()["name"] + "/cml-lang-self-checkpoints"

# 1) the run so far (checkpoints p000-p018 + logs), from the HF backup
if not glob.glob(P("runs", RUN, "ckpt", "p*.pt")):
    snapshot_download(HF_REPO, local_dir=P("runs"), allow_patterns=[f"{RUN}/*"])
ck = sorted(glob.glob(P("runs", RUN, "ckpt", "p*.pt")))
print(f"{RUN}: {len(ck)} checkpoints, last = {os.path.basename(ck[-1]) if ck else None}")

# 2) FLORES+ (gated: needs your HF login)
FLORES_CODE = {"en": "eng_Latn", "hi": "hin_Deva", "zh": "cmn_Hans", "ar": "arb_Arab"}
for l, code in FLORES_CODE.items():
    for split in ["dev", "devtest"]:
        p = P(f"data/flores/{split}.{l}.jsonl")
        if not os.path.exists(p):
            from datasets import load_dataset
            os.makedirs(os.path.dirname(p), exist_ok=True)
            ds = load_dataset("openlanguagedata/flores_plus", code, split=split)
            with open(p, "w", encoding="utf-8") as f:
                for r in ds:
                    f.write(json.dumps({"text": r["text"]}, ensure_ascii=False) + "\n")
            print("downloaded", p, len(ds))

# 3) training data: must already be here
missing = [f"data/en-{l}/{s}.jsonl" for l in LANGS for s in ["train", "val"] if not os.path.exists(P(f"data/en-{l}/{s}.jsonl"))]
print("training data OK" if not missing else f"MISSING: {missing}  -> copy them here before training (Arabic: arabic.ipynb rebuilds it)")

v4b_only_hi_s0: 19 checkpoints, last = p018_hi.pt
training data OK


In [3]:
C = RunConfig(run_name=RUN, mode="only_hi", order=("hi", "zh", "ar"), n_phases=90, seed=0,
              time_cap_h=120.0, hf_repo=HF_REPO)
print(C)
nxt = max(int(os.path.basename(p)[1:4]) for p in ck) + 1 if ck else 0
print(f"will resume at phase {nxt} ({phase_lang(C, nxt) if nxt else 'init'}) and stop after phase {C.n_phases}")

RunConfig(run_name='v4b_only_hi_s0', mode='only_hi', order=('hi', 'zh', 'ar'), init_ckpt=None, n_phases=90, seed=0, max_tokens=16000, lr=0.0002, min_lr=2e-05, warmup_steps=500, cooldown_steps=2000, min_steps=2000, max_stable_steps=60000, eval_every=500, plateau_window=4, plateau_tol=0.005, weight_decay=0.1, grad_clip=1.0, masked=True, diverge_factor=1.15, max_retries=2, time_cap_h=120.0, hf_repo='s8n29/cml-lang-self-checkpoints')
will resume at phase 19 (hi) and stop after phase 90


In [4]:
D = load_data(train_langs=["hi"])

  hi: 993,016 train / 2,000 val
  zh: 2,000 val
  ar: 2,000 val
data loaded in 20s


In [5]:
model = train_run(C, D)

 RUN v4b_only_hi_s0 | mode only_hi | 90 phases after phase 0 | device cuda
 phase order: init (random) → hi → hi → hi → hi → hi → hi → hi → ...
 each phase: fresh AdamW (wd 0.1 on matrices), warmup 500 → lr 0.0002 → plateau (<0.5% over 4-eval windows, after 2,000, cap 60,000) → cooldown 2,000 steps to 2e-05
 masked (script-restricted) loss & decoding: True | divergence guard: >1.15x best twice → retry at half lr (max 2)
RESUMING after phase 18 (hi)

  new best checkpoint: hi = phase 8 (chrF 50.1)
▶ PHASE 19/90 | training hi (993,016 pairs)


[W1015 00:52:19.559124365 CUDACachingAllocator.cpp:3934] memory allocation failed with OOM on device 0 while trying to allocate 2048917504 bytes (free: 1124073472, total: 25336807424).
[W1015 00:53:18.486634531 CUDACachingAllocator.cpp:3934] memory allocation failed with OOM on device 0 while trying to allocate 1952448512 bytes (free: 1501560832, total: 25336807424).


  [p19 hi] step    500 | lr 2.0e-04 | train 2.043 | val 1.356 (best 1.356) | window Δ n/a | stable | 76k tok/s | phase 1m39s | run ETA: after first phase
  [p19 hi] step  1,000 | lr 2.0e-04 | train 2.141 | val 1.353 (best 1.353) | window Δ n/a | stable | 79k tok/s | phase 3m12s | run ETA: after first phase
  [p19 hi] step  1,500 | lr 2.0e-04 | train 2.183 | val 1.350 (best 1.350) | window Δ n/a | stable | 79k tok/s | phase 4m45s | run ETA: after first phase
  [p19 hi] step  2,000 | lr 2.0e-04 | train 2.158 | val 1.369 (best 1.350) | window Δ n/a | stable | 77k tok/s | phase 6m21s | run ETA: after first phase
  [p19 hi] step  2,500 | lr 2.0e-04 | train 2.173 | val 1.353 (best 1.350) | window Δ n/a | stable | 79k tok/s | phase 7m54s | run ETA: after first phase
  [p19 hi] step  3,000 | lr 2.0e-04 | train 2.188 | val 1.351 (best 1.350) | window Δ n/a | stable | 79k tok/s | phase 9m26s | run ETA: after first phase
  [p19 hi] step  3,500 | lr 2.0e-04 | train 2.179 | val 1.359 (best 1.350) |

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

...

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Validating:           |   0%            

No files have been modified since last commit. Skipping to prevent empty commit.


  [p90 hi] step    500 | lr 2.0e-04 | train 1.902 | val 1.377 (best 1.377) | window Δ n/a | stable | 75k tok/s | phase 1m38s | run ETA ~19m19s
  [p90 hi] step  1,000 | lr 2.0e-04 | train 2.016 | val 1.366 (best 1.366) | window Δ n/a | stable | 76k tok/s | phase 3m14s | run ETA ~19m19s
  [p90 hi] step  1,500 | lr 2.0e-04 | train 2.060 | val 1.359 (best 1.359) | window Δ n/a | stable | 76k tok/s | phase 4m50s | run ETA ~19m19s
  [p90 hi] step  2,000 | lr 2.0e-04 | train 2.036 | val 1.378 (best 1.359) | window Δ n/a | stable | 75k tok/s | phase 6m27s | run ETA ~19m19s
  [p90 hi] step  2,500 | lr 2.0e-04 | train 2.048 | val 1.371 (best 1.359) | window Δ n/a | stable | 78k tok/s | phase 8m01s | run ETA ~19m19s
  [p90 hi] step  3,000 | lr 2.0e-04 | train 2.075 | val 1.359 (best 1.359) | window Δ n/a | stable | 78k tok/s | phase 9m35s | run ETA ~19m19s
  [p90 hi] step  3,500 | lr 2.0e-04 | train 2.059 | val 1.371 (best 1.359) | window Δ n/a | stable | 77k tok/s | phase 11m11s | run ETA ~19m19

In [6]:
print_run_summary(RUN)

 SUMMARY v4b_only_hi_s0: 91 phases (incl. phase 0)
--------------------------------------------------------------------------------------
  ph lang   steps      stop | chrF hi chrF zh chrF ar | val hi val zh val ar   (script-restricted)
   0 init       0      init |     1.2     0.0     2.7 |  8.965  9.641  9.196
   1   hi  33,000   plateau |    49.1     0.4     0.4 |  1.356  8.905  8.440
   2   hi   6,000   plateau |    49.4     0.5     0.5 |  1.345  8.952  8.444
   3   hi   6,000   plateau |    49.3     0.5     0.5 |  1.337  8.989  8.447
   4   hi   6,000   plateau |    49.7     0.5     0.5 |  1.329  8.969  8.418
   5   hi   6,000   plateau |    49.7     0.5     0.6 |  1.322  8.955  8.435
   6   hi   6,000   plateau |    49.2     0.5     0.6 |  1.318  9.003  8.469
   7   hi   6,000   plateau |    49.5     0.5     0.6 |  1.315  8.949  8.421
   8   hi   6,000   plateau |    50.1     0.5     0.6 |  1.311  8.997  8.442
   9   hi   6,000 worsening |    49.2     0.6     0.8 |  1.310  9.048 